In [ ]:
!pip install -q polars
!pip install -q lightgbm
from pathlib import Path
import glob
import os
import sys
import math
import joblib
import json
import numpy as np
import pandas as pd
import polars as pl
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error  
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


FIELD_LENGTH = 120.0
FIELD_WIDTH  = 53.3


Input files: 18
Output files: 18
------------ INPUT DF INFO ------------ 
<class 'pandas.DataFrame'>
RangeIndex: 4880579 entries, 0 to 4880578
Data columns (total 23 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   game_id                   4880579 non-null  int64  
 1   play_id                   4880579 non-null  int64  
 2   player_to_predict         4880579 non-null  bool   
 3   nfl_id                    4880579 non-null  int64  
 4   frame_id                  4880579 non-null  int64  
 5   play_direction            4880579 non-null  object 
 6   absolute_yardline_number  4880579 non-null  int64  
 7   player_name               4880579 non-null  object 
 8   player_height             4880579 non-null  object 
 9   player_weight             4880579 non-null  int64  
 10  player_birth_date         4880579 non-null  object 
 11  player_position           4880579 non-null  object 
 12  player_side          

ModuleNotFoundError: No module named 'kaggle_evaluation'

In [ ]:

# Pick DATA_DIR based on environment
if os.path.exists("/kaggle/input/nfl-big-data-bowl-2026-prediction"):
    DATA_DIR = Path("/kaggle/input/nfl-big-data-bowl-2026-prediction")
else:
    DATA_DIR = Path(r"D:/2nd  year/1st term/Machine learning/project/nfl-big-data-bowl-2026-prediction")

# Prep training files
train_dir = DATA_DIR / "train"
input_files  = sorted(glob.glob(str(train_dir / "input_*.csv")))
output_files = sorted(glob.glob(str(train_dir / "output_*.csv")))
print(f"Input files: {len(input_files)}")
print(f"Output files: {len(output_files)}")
input_df = pd.concat([pd.read_csv(f) for f in input_files], ignore_index=True)
output_df = pd.concat([pd.read_csv(f) for f in output_files], ignore_index=True)
test_data = pd.read_csv(str(DATA_DIR / 'test.csv'))
print("------------ INPUT DF INFO ------------ ")
input_df.info(show_counts=True)
print(" ------------ OUTPUT DF INFO ------------  ")
output_df.info(show_counts=True)

In [ ]:
# Feature engineering

def uniform_play(df: pd.DataFrame) -> pd.DataFrame:
    """Mirror left-going plays so all plays go right; adjust coords & angles."""
    df = df.copy()
    mask = df['play_direction'].astype(str).str.lower().eq('left')

    # Mirror field coordinates (length 120, width 53.3)
    df.loc[mask, 'x'] = 120 - df.loc[mask, 'x']
    df.loc[mask, 'y'] = 53.3 - df.loc[mask, 'y']

    # Mirror ball landing coords if present
    for c in ['ball_land_x', 'ball_land_y']:
        if c in df.columns:
            if c.endswith('_x'):
                df.loc[mask, c] = 120 - df.loc[mask, c]
            else:
                df.loc[mask, c] = 53.3 - df.loc[mask, c]

    # Mirror angles: dir & o are degrees in [0,360]
    for ang in ['dir', 'o']:
        if ang in df.columns:
            df.loc[mask, ang] = (180 - df.loc[mask, ang]) % 360

    # If vx/vy/ax/ay already exist, flip x-components
    for comp in ['vx', 'ax']:
        if comp in df.columns:
            df.loc[mask, comp] = -df.loc[mask, comp]
    # y-components unchanged under left-right mirror

    return df


def angle_diff_deg(a, b):
    """
    Smallest signed difference between two angles in degrees.
    a, b can be pandas Series or numpy arrays.
    Result in (-180, 180].
    """
    return (a - b + 180.0) % 360.0 - 180.0


def create_advanced_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create comprehensive feature set for optimal prediction"""
    df = df.copy()

    # Ball-related features
    df['dist_to_ball_land'] = np.sqrt(
        (df['x'] - df['ball_land_x'])**2 +
        (df['y'] - df['ball_land_y'])**2
    )

    df['angle_to_ball'] = np.arctan2(
        df['ball_land_y'] - df['y'],
        df['ball_land_x'] - df['x']
    )

    df['angle_to_ball_deg'] = np.degrees(df['angle_to_ball'])

    df['speed_to_ball'] = df['s'] * np.cos(
        np.radians(df['dir']) - df['angle_to_ball']
    )

    df['delta_x_to_ball'] = df['ball_land_x'] - df['x']
    df['delta_y_to_ball'] = df['ball_land_y'] - df['y']
    df['manhattan_dist_to_ball'] = np.abs(df['delta_x_to_ball']) + np.abs(df['delta_y_to_ball'])
    df['eucl_dist_to_ball'] = np.sqrt(df['delta_x_to_ball']**2 + df['delta_y_to_ball']**2)
    df['x_y_dist_ratio'] = np.abs(df['delta_x_to_ball']) / (np.abs(df['delta_y_to_ball']) + 0.1)

    # Extra ball-direction features
    df['ball_dx'] = df['ball_land_x'] - df['x']
    df['ball_dy'] = df['ball_land_y'] - df['y']
    df['dist_to_ball'] = np.sqrt(df['ball_dx']**2 + df['ball_dy']**2)
    df['ball_unit_x'] = df['ball_dx'] / (df['dist_to_ball'] + 1e-6)
    df['ball_unit_y'] = df['ball_dy'] / (df['dist_to_ball'] + 1e-6)

    angle_to_ball = np.arctan2(df['ball_dy'], df['ball_dx'] + 1e-6)
    df['angle_to_ball_cos'] = np.cos(angle_to_ball)
    df['angle_to_ball_sin'] = np.sin(angle_to_ball)

    # role / side flags
    role_map = {
        'Targeted Receiver': 4,
        'Defensive Coverage': 3,
        'Other Route Runner': 2,
        'Passer': 1
    }
    df['role_encoded'] = df['player_role'].map(role_map).fillna(0)
    df['is_offense'] = (df['player_side'] == 'Offense').astype(int)
    df['is_defense'] = (df['player_side'] == 'Defense').astype(int)
    df['is_targeted'] = (df['player_role'] == 'Targeted Receiver').astype(int)
    df['is_target_receiver'] = df['is_targeted']
    df['is_defender'] = (df['player_role'] == 'Defensive Coverage').astype(int)
    df['is_passer'] = (df['player_role'] == 'Passer').astype(int)

    # Field position
    df['field_position'] = df['absolute_yardline_number']

    df['dist_from_left_sideline'] = df['y']
    df['dist_from_right_sideline'] = 53.3 - df['y']
    df['dist_from_nearest_sideline'] = np.minimum(
        df['dist_from_left_sideline'],
        df['dist_from_right_sideline']
    )

    # Velocity and acceleration components
    df['vx'] = df['s'] * np.cos(np.radians(df['dir']))
    df['vy'] = df['s'] * np.sin(np.radians(df['dir']))
    df['ax'] = df['a'] * np.cos(np.radians(df['dir']))
    df['ay'] = df['a'] * np.sin(np.radians(df['dir']))
    df['speed_squared'] = df['s'] ** 2
    df['accel_squared'] = df['a'] ** 2

    # Orientation features
    df['orientation_dir_diff'] = np.abs(df['o'] - df['dir'])
    df['orientation_dir_diff'] = np.where(
        df['orientation_dir_diff'] > 180,
        360 - df['orientation_dir_diff'],
        df['orientation_dir_diff']
    )
    df['facing_ball'] = (df['orientation_dir_diff'] < 90).astype(int)

    # Physics-based features
    df['expected_time_to_ball'] = df['dist_to_ball_land'] / (df['s'] + 0.1)
    df['frames_after_ball'] = df['num_frames_output'] - (df['expected_time_to_ball'] * 10)

    df['estimated_x_next'] = df['x'] + df['vx'] * 0.1
    df['estimated_y_next'] = df['y'] + df['vy'] * 0.1
    df['estimated_x_1sec'] = df['x'] + df['vx']
    df['estimated_y_1sec'] = df['y'] + df['vy']

    df['can_reach_ball'] = (df['expected_time_to_ball'] < df['num_frames_output'] * 0.1).astype(int)

    # Kinetic energy proxy
    df['kinetic_energy'] = 0.5 * df['speed_squared']

    # Trajectory curvature proxy
    df['velocity_angle'] = np.arctan2(df['vy'], df['vx'])
    df['trajectory_alignment'] = np.abs(df['velocity_angle'] - df['angle_to_ball'])

    # "Facing ball" difference using heading vs ball ray
    heading_angle = np.arctan2(df['vy'], df['vx'] + 1e-9)
    heading_deg = np.degrees(heading_angle)
    ball_deg = np.degrees(angle_to_ball)
    df['facing_ball_diff'] = angle_diff_deg(heading_deg, ball_deg)

    # Play direction flag (before mirroring info)
    df['play_dir_flag'] = df['play_direction'].astype(str).str.lower().eq('left').astype(int)

    # --- Closeness / rank to ball ---
    play_group = df.groupby(['game_id', 'play_id'])['dist_to_ball_land']
    df['min_dist_to_ball_play'] = play_group.transform('min')
    df['dist_rank_in_play'] = play_group.rank(method='dense') - 1
    df['is_closest_to_ball'] = (df['dist_to_ball_land'] == df['min_dist_to_ball_play']).astype(int)

    side_group = df.groupby(['game_id', 'play_id', 'player_side'])['dist_to_ball_land']
    df['min_dist_to_ball_side'] = side_group.transform('min')
    df['is_closest_on_side'] = (df['dist_to_ball_land'] == df['min_dist_to_ball_side']).astype(int)

    return df


def add_sequence_features(df: pd.DataFrame, window: int = 3) -> pd.DataFrame:
    """
    Add temporal sequence features + lags + angle encodings + frame time features
    + context features (last/prev/lag2, jerk, team clustering, QB, etc.)
    """
    df = df.sort_values(['game_id', 'play_id', 'nfl_id', 'frame_id']).copy()

    g = df.groupby(['game_id', 'play_id', 'nfl_id'])

    # --- 1) Lagged kinematics & positions ---

    # Lag-1
    df['x_prev'] = g['x'].shift(1)
    df['y_prev'] = g['y'].shift(1)
    df['s_prev'] = g['s'].shift(1)
    df['a_prev'] = g['a'].shift(1)
    df['dir_prev'] = g['dir'].shift(1)
    df['o_prev'] = g['o'].shift(1)
    df['vx_prev'] = g['vx'].shift(1)
    df['vy_prev'] = g['vy'].shift(1)

    # Lag-2
    df['x_lag2'] = g['x'].shift(2)
    df['y_lag2'] = g['y'].shift(2)
    df['s_lag2'] = g['s'].shift(2)
    df['a_lag2'] = g['a'].shift(2)
    df['dir_lag2'] = g['dir'].shift(2)
    df['o_lag2'] = g['o'].shift(2)
    df['vx_lag2'] = g['vx'].shift(2)
    df['vy_lag2'] = g['vy'].shift(2)

    # Displacement deltas
    df['dx_prev'] = df['x'] - df['x_prev']
    df['dy_prev'] = df['y'] - df['y_prev']
    df['dx_lag2'] = df['x_prev'] - df['x_lag2']
    df['dy_lag2'] = df['y_prev'] - df['y_lag2']

    # --- Delta features (frame-to-frame changes) ---
    df['delta_vx']    = g['vx'].diff()
    df['delta_vy']    = g['vy'].diff()
    df['delta_speed'] = g['s'].diff()
    df['delta_ax']    = g['ax'].diff()
    df['delta_ay']    = g['ay'].diff()

    # Frame deltas
    df['frame_prev'] = g['frame_id'].shift(1)
    df['frame_lag2'] = g['frame_id'].shift(2)
    df['prev_dt'] = (df['frame_id'] - df['frame_prev']).replace(0, 1)
    df['lag2_dt'] = (df['frame_prev'] - df['frame_lag2']).replace(0, 1)

    # Speed estimates from lagged velocities
    df['prev_speed_est'] = np.sqrt(df['vx_prev']**2 + df['vy_prev']**2)
    df['lag2_speed_est'] = np.sqrt(df['vx_lag2']**2 + df['vy_lag2']**2)

    # First differences in speed / acceleration
    df['delta_s'] = df['s'] - df['s_prev']
    df['delta_a'] = df['a'] - df['a_prev']

    # Angle changes
    df['dir_change'] = angle_diff_deg(df['dir'], df['dir_prev'])
    df['o_change'] = angle_diff_deg(df['o'], df['o_prev'])

    # --- 2) Rolling features ---
    for col in ['s', 'a', 'vx', 'vy']:
        df[f'{col}_rolling_mean'] = g[col].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )
        df[f'{col}_rolling_std'] = g[col].transform(
            lambda x: x.rolling(window=window, min_periods=1).std()
        )

    # --- 3) Angle encodings for dir and o (current & previous) ---
    rad_dir = np.radians(df['dir'].fillna(0.0))
    rad_o = np.radians(df['o'].fillna(0.0))
    df['dir_sin'] = np.sin(rad_dir)
    df['dir_cos'] = np.cos(rad_dir)
    df['o_sin']   = np.sin(rad_o)
    df['o_cos']   = np.cos(rad_o)

    rad_dir_prev = np.radians(df['dir_prev'].fillna(df['dir']))
    rad_o_prev = np.radians(df['o_prev'].fillna(df['o']))
    df['dir_prev_sin'] = np.sin(rad_dir_prev)
    df['dir_prev_cos'] = np.cos(rad_dir_prev)
    df['o_prev_sin']   = np.sin(rad_o_prev)
    df['o_prev_cos']   = np.cos(rad_o_prev)

    # current-as-last encodings
    df['dir_last'] = df['dir']
    df['o_last'] = df['o']
    df['dir_last_sin'] = df['dir_sin']
    df['dir_last_cos'] = df['dir_cos']
    df['o_last_sin'] = df['o_sin']
    df['o_last_cos'] = df['o_cos']

    # --- 4) Frame timing features ---
    df['frame_ratio'] = df['frame_id'] / (df['num_frames_output'] + 1e-3)
    df['frame_time']  = df['frame_id'] * 0.1    # 10 frames per second
    df['frame_id_sq'] = df['frame_id']**2
    df['time_to_ball'] = df['num_frames_output'] * 0.1

    # --- 5) "Last observed" per sequence ---
    df['last_observed_frame'] = g['frame_id'].transform('max')
    df['observed_duration'] = df['last_observed_frame'] / 10.0 

    # For "last" state features, take group-wise last of kinematics
    df['x_last'] = g['x'].transform('last')
    df['y_last'] = g['y'].transform('last')
    df['s_last'] = g['s'].transform('last')
    df['a_last'] = g['a'].transform('last')
    df['dir_last'] = g['dir'].transform('last')
    df['o_last'] = g['o'].transform('last')

    # Last-frame velocity components
    last_dir_rad = np.radians(df['dir_last'].fillna(0.0))
    df['vel_x_last'] = df['s_last'].fillna(0.0) * np.cos(last_dir_rad)
    df['vel_y_last'] = df['s_last'].fillna(0.0) * np.sin(last_dir_rad)

    # Previous velocity and acceleration at last frame
    df['vel_x_prev'] = df['vx_prev']
    df['vel_y_prev'] = df['vy_prev']
    df['acc_x_last'] = df['vel_x_last'] - df['vel_x_prev']
    df['acc_y_last'] = df['vel_y_last'] - df['vel_y_prev']

    # Speed & acceleration ratios
    df['speed_ratio'] = df['s_last'] / (np.abs(df['prev_speed_est']) + 1e-3)
    df['acc_ratio'] = df['a_last'] / (np.abs(df['a_prev']) + 1e-3)

    # Jerk estimate
    df['jerk_est'] = (df['delta_s'] / (df['prev_dt'] + 1e-3)) * 10.0

    # --- 6) Group-level context (nearest defender, clustering, QB, etc.) ---
    df['nearest_defender_dist'] = np.nan
    df['nearest_defender_speed'] = np.nan
    df['off_cluster_density'] = np.nan
    df['def_cluster_density'] = np.nan
    df['off_mean_s'] = np.nan
    df['def_mean_s'] = np.nan
    df['num_def_within_5'] = np.nan
    df['num_off_within_5'] = np.nan
    df['dist_to_qb'] = np.nan

    # Last frame mask per play
    last_frame_per_play = df.groupby(['game_id', 'play_id'])['frame_id'].transform('max')
    last_mask = df['frame_id'] == last_frame_per_play

    for (g_id, p_id), grp in df[last_mask].groupby(['game_id', 'play_id']):
        idx = grp.index
        off_idx = grp.index[grp['player_side'] == 'Offense'].tolist()
        def_idx = grp.index[grp['player_side'] == 'Defense'].tolist()

        # Mean team speeds
        if off_idx:
            off_mean = grp.loc[off_idx, 's'].mean()
            df.loc[off_idx, 'off_mean_s'] = off_mean
        if def_idx:
            def_mean = grp.loc[def_idx, 's'].mean()
            df.loc[off_idx, 'def_mean_s'] = def_mean  # broadcast to offense

        # Offense cluster density & num_off_within_5
        if len(off_idx) > 1:
            off_coords = grp.loc[off_idx, ['x', 'y']].values
            off_pair = off_coords[:, None, :] - off_coords[None, :, :]
            off_dists = np.sqrt((off_pair ** 2).sum(axis=2))
            mean_pair = off_dists.mean(axis=1)
            df.loc[off_idx, 'off_cluster_density'] = mean_pair
            num_close_off = (off_dists < 5).sum(axis=1) - 1
            df.loc[off_idx, 'num_off_within_5'] = num_close_off

        # Defense cluster density
        if len(def_idx) > 1:
            def_coords = grp.loc[def_idx, ['x', 'y']].values
            def_pair = def_coords[:, None, :] - def_coords[None, :, :]
            def_dists = np.sqrt((def_pair ** 2).sum(axis=2))
            mean_pair = def_dists.mean(axis=1)
            df.loc[def_idx, 'def_cluster_density'] = mean_pair

        # Nearest defender & num_def_within_5 for offense players
        if off_idx and def_idx:
            off_coords = grp.loc[off_idx, ['x', 'y']].values
            def_coords = grp.loc[def_idx, ['x', 'y']].values
            diff = off_coords[:, None, :] - def_coords[None, :, :]
            dists = np.sqrt((diff ** 2).sum(axis=2))

            min_idx = dists.argmin(axis=1)
            min_val = dists[np.arange(len(off_idx)), min_idx]
            df.loc[off_idx, 'nearest_defender_dist'] = min_val
            df.loc[off_idx, 'nearest_defender_speed'] = grp.loc[def_idx, 's'].values[min_idx]

            num_close_def = (dists < 5).sum(axis=1)
            df.loc[off_idx, 'num_def_within_5'] = num_close_def

        # Distance to QB for offense
        qb_idx = grp.index[grp['player_position'] == 'QB'].tolist()
        if qb_idx and off_idx:
            qb_pos = grp.loc[qb_idx[0], ['x', 'y']].values.astype(float)
            off_coords = grp.loc[off_idx, ['x', 'y']].values.astype(float)
            dqb = np.sqrt(((off_coords - qb_pos) ** 2).sum(axis=1))
            df.loc[off_idx, 'dist_to_qb'] = dqb

    # Target-specific last-speed
    df['target_s_last'] = np.where(df.get('is_target_receiver', 0) == 1, df['s_last'], 0.0)

    df.fillna(0, inplace=True)
    return df


def height_to_inches(h):
    try:
        f, i = h.split("-")
        return int(f) * 12 + int(i)
    except:
        return np.nan


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # convert height once, for both train and test
    if 'player_height' in df.columns:
        df['player_height'] = df['player_height'].apply(height_to_inches)
    df = uniform_play(df)
    df = create_advanced_features(df)
    df = add_sequence_features(df)
    return df

In [ ]:
# apply to training input
engineered_df = engineer_features(input_df)
players2predict = engineered_df[engineered_df["player_to_predict"] == True].copy()
play_dir_info = (input_df[["game_id", "play_id", "nfl_id", "frame_id", "play_direction"]].drop_duplicates())
output_mirrored = output_df.merge(play_dir_info,
                                  on=["game_id", "play_id", "nfl_id", "frame_id"],
                                  how="left")
output_mirrored = uniform_play(output_mirrored)
merged = players2predict.merge(output_mirrored,
                               on=["game_id", "play_id", "nfl_id", "frame_id"],
                               suffixes=("_input", "_output"))
merged["dx"] = merged["x_output"] - merged["x_input"]
merged["dy"] = merged["y_output"] - merged["y_input"]
feature_cols = [
    'absolute_yardline_number', 'player_weight', 'player_height',
    'x_input', 'y_input', 'num_frames_output',
    'ball_land_x', 'ball_land_y',
    's', 'a', 'dir', 'o', 'vx', 'vy', 'ax', 'ay',
    'speed_squared', 'accel_squared',
    'dist_to_ball_land', 'angle_to_ball', 'angle_to_ball_deg',
    'speed_to_ball', 'delta_x_to_ball', 'delta_y_to_ball',
    'eucl_dist_to_ball', 'manhattan_dist_to_ball', 'x_y_dist_ratio',
    'ball_dx', 'ball_dy', 'dist_to_ball',
    'ball_unit_x', 'ball_unit_y',
    'angle_to_ball_cos', 'angle_to_ball_sin',
    'facing_ball_diff',
    'role_encoded', 'is_offense', 'is_defense',
    'is_targeted', 'is_defender', 'is_passer',
    'is_target_receiver',
    'field_position', 'play_dir_flag',
    'dist_from_left_sideline', 'dist_from_right_sideline',
    'dist_from_nearest_sideline',
    'expected_time_to_ball', 'frames_after_ball',
    'estimated_x_next', 'estimated_y_next',
    'estimated_x_1sec', 'estimated_y_1sec',
    'can_reach_ball', 'kinetic_energy',
    'velocity_angle', 'trajectory_alignment',
    'delta_vx', 'delta_vy', 'delta_speed',
    'delta_ax', 'delta_ay',
    's_rolling_mean', 's_rolling_std',
    'a_rolling_mean', 'a_rolling_std',
    'vx_rolling_mean', 'vx_rolling_std',
    'vy_rolling_mean', 'vy_rolling_std',
    'x_prev', 'y_prev', 's_prev', 'a_prev',
    'dir_prev', 'o_prev', 'vx_prev', 'vy_prev',
    'x_lag2', 'y_lag2', 's_lag2', 'a_lag2',
    'dir_lag2', 'o_lag2', 'vx_lag2', 'vy_lag2',
    'dx_prev', 'dy_prev', 'dx_lag2', 'dy_lag2',
    'prev_dt', 'lag2_dt',
    'prev_speed_est', 'lag2_speed_est',
    'delta_s', 'delta_a',
    'dir_change', 'o_change',
    'dir_sin', 'dir_cos', 'o_sin', 'o_cos',
    'dir_prev_sin', 'dir_prev_cos',
    'o_prev_sin', 'o_prev_cos',
    'dir_last_sin', 'dir_last_cos',
    'o_last_sin', 'o_last_cos',
    'x_last', 'y_last', 's_last', 'a_last',
    'dir_last', 'o_last',
    'vel_x_last', 'vel_y_last',
    'vel_x_prev', 'vel_y_prev',
    'acc_x_last', 'acc_y_last',
    'speed_ratio', 'acc_ratio', 'jerk_est',
    'frame_id', 'frame_time', 'frame_ratio', 'frame_id_sq',
    'time_to_ball', 'last_observed_frame', 'observed_duration',
    'min_dist_to_ball_play', 'dist_rank_in_play',
    'is_closest_to_ball', 'min_dist_to_ball_side',
    'is_closest_on_side',
    'nearest_defender_dist', 'nearest_defender_speed',
    'off_cluster_density', 'def_cluster_density',
    'off_mean_s', 'def_mean_s',
    'num_def_within_5', 'num_off_within_5',
    'dist_to_qb',
    'target_s_last',
]
X = merged[feature_cols].fillna(0)
y = merged[["dx", "dy"]]

# Split offense / defense
offense_df = merged[merged["is_offense"] == 1].copy()
defense_df = merged[merged["is_offense"] == 0].copy()

X_off = offense_df[feature_cols].fillna(0)
y_off_dx = offense_df["dx"]
y_off_dy = offense_df["dy"]

X_def = defense_df[feature_cols].fillna(0)
y_def_dx = defense_df["dx"]
y_def_dy = defense_df["dy"]
# Train LightGBM models
lgb_params = dict(
    objective="regression",
    boosting_type="gbdt",
    learning_rate=0.1,
    num_leaves=255,
    subsample=0.9,
    colsample_bytree=0.9,
    n_estimators=2000,    
    reg_alpha=0.01,
    reg_lambda=0.1,
    min_child_samples=10,
    n_jobs=-1,
    random_state=42,)
lgb_dx_off = LGBMRegressor(**lgb_params)
lgb_dy_off = LGBMRegressor(**lgb_params)
lgb_dx_off.fit(X_off, y_off_dx)
lgb_dy_off.fit(X_off, y_off_dy)
lgb_dx_def = LGBMRegressor(**lgb_params)
lgb_dy_def = LGBMRegressor(**lgb_params)
lgb_dx_def.fit(X_def, y_def_dx)
lgb_dy_def.fit(X_def, y_def_dy)

# ---- Save LGBM models ----
joblib.dump(lgb_dx_off, "lgb_dx_off.pkl")
joblib.dump(lgb_dy_off, "lgb_dy_off.pkl")
joblib.dump(lgb_dx_def, "lgb_dx_def.pkl")
joblib.dump(lgb_dy_def, "lgb_dy_def.pkl")

# ---- Save feature column list ----
with open("feature_cols.json", "w") as f:
    json.dump(feature_cols, f)


In [ ]:
# For kaggle submission
def predict(test: pl.DataFrame, test_input: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
    """
    Return a DataFrame with exactly two columns ['x','y'], same number of rows
    as `test` (row order preserved), using the trained LightGBM offense/defense
    displacement models.
    """
    base = test.to_pandas()
    df   = test_input.to_pandas()
    df = df[df['player_to_predict'] == True].copy()
    was_left = df['play_direction'].astype(str).str.lower().eq('left')
    df = engineer_features(df)
    df['x_input'] = df['x']
    df['y_input'] = df['y']
    X_api = df[feature_cols].fillna(0)
    preds_dx = np.zeros(len(X_api), dtype=float)
    preds_dy = np.zeros(len(X_api), dtype=float)
    off_mask = (df['is_offense'] == 1).values
    def_mask = ~off_mask
    if off_mask.any():
        preds_dx[off_mask] = lgb_dx_off.predict(X_api[off_mask])
        preds_dy[off_mask] = lgb_dy_off.predict(X_api[off_mask])
    if def_mask.any():
        preds_dx[def_mask] = lgb_dx_def.predict(X_api[def_mask])
        preds_dy[def_mask] = lgb_dy_def.predict(X_api[def_mask])
    x_pred = df['x_input'].values + preds_dx
    y_pred = df['y_input'].values + preds_dy
    left_idx = was_left.values
    x_pred[left_idx] = FIELD_LENGTH - x_pred[left_idx]
    y_pred[left_idx] = FIELD_WIDTH  - y_pred[left_idx]
    pred_df = df[['game_id', 'play_id', 'nfl_id', 'frame_id']].copy()
    pred_df['x'] = x_pred
    pred_df['y'] = y_pred
    out = base.merge(pred_df, on=['game_id', 'play_id', 'nfl_id', 'frame_id'],
                     how='left')[['x', 'y']].fillna(0)
    assert len(out) == len(base), "Output length must match test.csv length."
    return out
